#  프롬프트 엔지니어링 - Zero-shot, Few-shot

## 학습 목표

1. Zero-shot, One-shot, Few-shot 프롬프팅의 개념과 차이점을 이해한다
2. PromptTemplate과 partial 메소드를 사용하여 프롬프트를 구성할 수 있다
3. FewShotChatMessagePromptTemplate을 활용하여 고정 예시 기반 프롬프팅을 구현한다
4. SemanticSimilarityExampleSelector를 사용하여 동적 Few-shot 프롬프팅을 구현한다

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob

from pprint import pprint
import json

`(3) LLM 설정`

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4.1-nano',
    temperature=0.3,
    top_p=0.9
)

e:\sw\dev\ai\modu_llm7\faq_bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Ollama 설치 및 모델 준비 안내 (Dynamic Few-Shot 실습용)
print("""
[Ollama 설정 필수]
Dynamic Few-Shot 프롬프팅 실습을 위해 Ollama가 필요합니다.

1. Ollama 설치: https://ollama.com/
2. 터미널에서 임베딩 모델 다운로드:
   ollama pull bge-m3
""")


[Ollama 설정 필수]
Dynamic Few-Shot 프롬프팅 실습을 위해 Ollama가 필요합니다.

1. Ollama 설치: https://ollama.com/
2. 터미널에서 임베딩 모델 다운로드:
   ollama pull bge-m3



## Zero-shot 프롬프팅

- **Zero-shot 프롬프팅**은 예시 없이 AI가 즉시 작업을 수행하는 기법입니다

- 명확한 **지시사항**만으로 원하는 결과를 얻을 수 있어 **사용이 간단**합니다

- 단순하고 직관적인 작업에 적합한 프롬프팅 방식이지만, 작업의 **복잡도에 따라 선택적 사용**이 필요합니다

In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Zero-shot 프롬프트 템플릿 생성
zero_shot_prompt = PromptTemplate(
    input_variables=["topic"],
    template="다음 시장에서 삼성전자의 경쟁업체를 설명해주세요: {topic}"
)

# 체인 생성
chain = zero_shot_prompt | llm | StrOutputParser()

# Zero-shot 실행
topic = "인공지능 반도체"
zero_shot_result = chain.invoke(input={"topic": topic}) 

print(zero_shot_result)

인공지능 반도체 시장에서 삼성전자의 주요 경쟁업체는 다음과 같습니다:

1. **엔비디아 (NVIDIA)**  
   - 인공지능 및 딥러닝 분야에서 선도적인 그래픽 처리장치(GPU) 제조사입니다. 특히 딥러닝 가속화에 최적화된 GPU와 AI 플랫폼을 제공하며, 자율주행, 데이터센터, 클라우드 컴퓨팅 등 다양한 분야에서 강력한 입지를 갖추고 있습니다.

2. **인텔 (Intel)**  
   - CPU뿐만 아니라 AI 가속기 및 FPGA(필드 프로그래머블 게이트 어레이) 솔루션도 개발하고 있습니다. 인텔의 Movidius와 Habana Labs 인공지능 칩은 AI 처리 능력을 강화하는 데 사용됩니다.

3. **구글 (Google)**  
   - 자체 AI 칩인 TPU(Tensor Processing Unit)를 개발하여, 클라우드 기반 AI 서비스와 머신러닝 작업에 활용하고 있습니다. TPU는 특히 딥러닝 모델의 학습과 추론에 최적화되어 있습니다.

4. **중국 업체들 (예: 화웨이, 바이두, 알리바바)**  
   - 화웨이의 Ascend 시리즈, 바이두의 Kunlun 칩, 알리바바의 Hanguang 등 중국 기업들도 AI 반도체 개발에 적극 나서고 있습니다. 이들은 주로 클라우드와 데이터센터 시장을 타깃으로 하고 있습니다.

5. **마이크로소프트 (Microsoft)**  
   - 자체 AI 하드웨어 개발보다는, 엔비디아와 협력하거나 Azure 클라우드 플랫폼을 통해 AI 가속기 솔루션을 제공하며 경쟁하고 있습니다.

6. **기타 스타트업 및 전문 업체들**  
   - Graphcore, Cerebras, Habana Labs(인텔 인수), Mythic 등 다양한 스타트업들이 혁신적인 AI 칩을 개발하여 시장 경쟁에 참여하고 있습니다.

이처럼 인공지능 반도체 시장은 기술력과 시장 점유율을 놓고 글로벌 기업들이 치열하게 경쟁하는 분야입니다. 삼성전자는 강력한 메모리 반도체 기술과 시스템 반도체 설계 역량을 바탕으로 이 시장에서도 경쟁력을 강화

In [6]:
# Zero-shot 프롬프팅 - 컨텍스트(Context) 제공 
zero_shot_prompt = PromptTemplate(
    input_variables=["context", "topic"],
    template="""{topic} 시장에서 삼성전자의 경쟁업체를 설명해주세요. 
    반드시 다음 제시된 뉴스에 근거해서 답변하세요:

    [뉴스]
    {context}
    
    [답변]
    """
)


# 체인 생성
chain = zero_shot_prompt | llm | StrOutputParser()

# Zero-shot 실행
context = """삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다. 
이는 AI 반도체 시장에서 지배적인 위치를 차지하고 있는 엔비디아의 독점을 도전하고, 
세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 삼성전자의 노력으로 해석된다.
"""

topic = "인공지능 반도체"
zero_shot_result = chain.invoke(input={"context": context, "topic": topic})

print(zero_shot_result)

삼성전자의 인공지능 반도체 시장 경쟁업체는 주로 엔비디아입니다. 뉴스에 따르면, 삼성전자는 내년 초에 자체 개발한 AI 가속기를 처음으로 출시할 예정으로, 이는 AI 반도체 시장에서 엔비디아의 독점적 지위에 도전하려는 전략입니다. 엔비디아는 이미 AI 가속기 시장에서 지배적인 위치를 차지하고 있으며, 많은 AI 및 딥러닝 응용 분야에서 표준으로 자리 잡고 있습니다. 삼성전자는 이러한 경쟁 구도 속에서 세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 노력을 기울이고 있으며, 이번 AI 가속기 출시를 통해 시장 점유율 확대와 경쟁력 강화를 목표로 하고 있습니다.


## One-shot 프롬프팅

- **One-shot 프롬프팅**은 하나의 예시를 통해 AI가 작업 패턴을 학습하는 기법입니다

- **Zero-shot** 방식보다 더 나은 성능을 제공하며, **형식화된 작업**에 특히 효과적입니다

- 단일 예시로 **품질 향상**이 가능하나, 해당 예시에 **과의존**할 수 있는 한계가 있습니다

In [7]:
### One-shot 프롬프트 템플릿 생성
# 1. Zero-shot 프롬프트 템플릿에 예시(example)를 포함하도록 수정
# 2. input_variables에 example_topic과 example_response 추가

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

one_shot_prompt = PromptTemplate(
    input_variables=["example_topic", "example_response", "topic"],
    template="""다음은 특정 시장에서 삼성전자의 경쟁업체를 설명하는 예시이다:

시장: {example_topic}
경쟁업체: {example_response}

이제 다음 시장에서 삼성전자의 경쟁업체를 설명해주세요:
시장: {topic}"""
)

# Example 데이터 설정
example_topic = "스마트폰"
example_response = """애플: 프리미엄 시장에서 주요 경쟁사로, iPhone 시리즈로 경쟁
샤오미: 중저가 시장에서 강세를 보이며 글로벌 시장 점유율 확대
구글: Pixel 시리즈로 프리미엄 시장 진출, AI 기능 강조"""

`(1) PromptTemplate 그대로 사용`

In [8]:
# one_shot_prompt 적용한 체인 생성
chain = one_shot_prompt | llm | StrOutputParser()

# One-shot 실행
topic = "인공지능 반도체"
one_shot_result = chain.invoke(
    input={
        "example_topic": example_topic,
        "example_response": example_response,
        "topic": topic
    }
)

print(f"one_shot_result:")
print(one_shot_result)

one_shot_result:
시장: 인공지능 반도체  
경쟁업체: 엔비디아: GPU 기반의 AI 가속기 시장에서 선도적 위치, 딥러닝 및 데이터센터용 솔루션 제공  
구글: TPU(텐서 프로세서 유닛)를 개발하여 클라우드 AI 서비스와 데이터센터용 인공지능 가속기 시장에서 경쟁  
인텔: 인공지능용 하드웨어와 칩셋 개발에 투자하며, 데이터센터 및 엣지 컴퓨팅 시장에서 경쟁력 확보  
마이크로소프트: 자체 AI 칩인 Azure AI 가속기 개발 및 클라우드 서비스와 연계하여 시장 확대


`(2) partial 메소드 사용`

In [11]:
# Example 데이터를 반영한 부분 프롬프트 출력 
partial_prompt = one_shot_prompt.partial(
    example_topic=example_topic,
    example_response=example_response,
)

print(f"partial_prompt:")
print(partial_prompt)

# 체인 생성
chain = partial_prompt | llm | StrOutputParser()

# One-shot 실행
topic = "인공지능 반도체"
one_shot_result = chain.invoke(input={"topic": topic})

print(f"one_shot_result:")
print(one_shot_result)

partial_prompt:
input_variables=['topic'] input_types={} partial_variables={'example_topic': '스마트폰', 'example_response': '애플: 프리미엄 시장에서 주요 경쟁사로, iPhone 시리즈로 경쟁\n샤오미: 중저가 시장에서 강세를 보이며 글로벌 시장 점유율 확대\n구글: Pixel 시리즈로 프리미엄 시장 진출, AI 기능 강조'} template='다음은 특정 시장에서 삼성전자의 경쟁업체를 설명하는 예시이다:\n\n시장: {example_topic}\n경쟁업체: {example_response}\n\n이제 다음 시장에서 삼성전자의 경쟁업체를 설명해주세요:\n시장: {topic}'
one_shot_result:
시장: 인공지능 반도체  
경쟁업체: 엔비디아: GPU 기반 인공지능 연산에 강점을 가지며, 데이터 센터와 자율주행 분야에서 시장 선도  
구글: TPU(텐서 프로세서 유닛) 개발로 구글 클라우드와 AI 서비스에 최적화된 솔루션 제공  
인텔: CPU와 함께 인공지능 가속기 개발에 투자하며, 산업용 및 데이터 센터용 인공지능 반도체 시장에서 경쟁  
마이크로소프트: 자사 클라우드 서비스와 연계한 AI 가속기 개발로 시장 확대 추진


## Few-shot 프롬프팅

- **Few-shot 프롬프팅**은 AI 모델에게 2-5개의 예시를 제공하여 학습시키는 방법입니다

- 이 방식은 **Zero-shot**이나 **One-shot** 프롬프팅보다 더 우수한 성능을 보여주며, 복잡한 작업에서 특히 효과적입니다

- Few-shot 프롬프팅은 높은 성능을 제공하지만, 긴 프롬프트로 인한 **비용 증가**를 고려해야 합니다

`(1) PromptTemplate 사용`

In [9]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Few-shot 프롬프트 템플릿 생성 
few_shot_prompt = PromptTemplate(
   input_variables=["examples", "topic"],
   template="""다음은 여러 시장에서 삼성전자의 경쟁업체를 설명하는 예시들이다:

{examples}

이제 다음 시장에서 삼성전자의 경쟁업체를 설명해주세요:
시장: {topic}"""
)

# Example 데이터 준비
examples = """
시장: 스마트폰
경쟁업체: 
- 애플(미국): 프리미엄 시장 주도, iPhone으로 경쟁
- 샤오미(중국): 중저가 시장 강세, 글로벌 확장중
- 구글(미국): Pixel로 AI 기능 강조

시장: TV
경쟁업체:
- LG전자(한국): OLED 기술 경쟁
- Sony(일본): 프리미엄 시장 경쟁
- TCL(중국): 중저가 시장 공략
"""

# 체인 생성 및 실행
chain = few_shot_prompt | llm | StrOutputParser()
result = chain.invoke({
   "examples": examples,
   "topic": "인공지능 반도체"
})

print(result)

시장: 인공지능 반도체  
경쟁업체:  
- 엔비디아(미국): GPU 기반 AI 가속기 선도, 딥러닝 분야 강자  
- AMD(미국): 고성능 GPU와 CPU 통합 솔루션 제공, AI 시장 확대 중  
- 인텔(미국): AI 전용 칩 개발, 데이터 센터 및 엣지 컴퓨팅에 집중  
- 화웨이(중국): AI 칩 개발 및 5G와 연계한 인공지능 솔루션 제공


`(2) partial 메소드 사용`

In [12]:
# partial 메서드를 사용하여 Few-shot 프롬프트 템플릿 생성
partial_prompt = few_shot_prompt.partial(
    examples=examples
)

print(f"partial_prompt:")
print(partial_prompt)

partial_prompt:
input_variables=['topic'] input_types={} partial_variables={'examples': '\n시장: 스마트폰\n경쟁업체: \n- 애플(미국): 프리미엄 시장 주도, iPhone으로 경쟁\n- 샤오미(중국): 중저가 시장 강세, 글로벌 확장중\n- 구글(미국): Pixel로 AI 기능 강조\n\n시장: TV\n경쟁업체:\n- LG전자(한국): OLED 기술 경쟁\n- Sony(일본): 프리미엄 시장 경쟁\n- TCL(중국): 중저가 시장 공략\n'} template='다음은 여러 시장에서 삼성전자의 경쟁업체를 설명하는 예시들이다:\n\n{examples}\n\n이제 다음 시장에서 삼성전자의 경쟁업체를 설명해주세요:\n시장: {topic}'


In [13]:
# 체인 생성
chain = partial_prompt | llm | StrOutputParser()

# Few-shot 실행
topic = "인공지능 반도체"
few_shot_result = chain.invoke(input={"topic": topic})

print(f"few_shot_result:")
print(few_shot_result)

few_shot_result:
시장: 인공지능 반도체  
경쟁업체:  
- 엔비디아(미국): GPU 기반 인공지능 가속기 선도, 데이터센터와 자율주행 분야 강세  
- AMD(미국): 고성능 CPU와 GPU를 통한 인공지능 처리 능력 강화, 경쟁력 확보  
- 인텔(미국): AI 전용 칩 개발 및 데이터센터용 솔루션 확대, 산업 전반에 적용  
- 구글(미국): TPU(Tensor Processing Unit) 개발로 AI 연산 최적화, 클라우드 서비스와 연계  
- 화웨이(중국): 인공지능 반도체 개발 및 글로벌 시장 확대, 통신 및 모바일 분야와 연계


`(3) FewShotChatMessagePromptTemplate 사용`

* FewShotChatMessagePromptTemplate는 LangChain에서 제공하는 템플릿으로, **미리 정의된 고정된 예제들(Fixed Examples)** 을 프롬프트에 포함시켜 모델이 일관된 형식과 품질의 응답을 생성할 수 있도록 돕습니다.

* 이 방식은 특히 특정 형식이나 구조를 가진 출력이 필요한 경우(예: JSON 형식, 특정 분석 리포트 형식 등) 매우 유용하며, 예제들이 고정되어 있어 결과의 일관성을 보장할 수 있습니다.

* 단, 고정된 예제를 사용하기 때문에 상황에 따라 유연하게 대응하기 어려울 수 있으며, 모든 케이스를 커버하기 위해서는 신중한 예제 선택이 필요합니다.

In [17]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from textwrap import dedent # text의 모든 줄에서 같은 선행 공백을 제거하는 함수

# 예시 데이터 정의 : 뉴스 텍스트(input) + 키워드 추출 결과 (output)
examples = [
    {
        "input": dedent("""
                        정부는 의과대학 입학 정원을 2000명 증가시킬 계획의 세부사항을 이달 20일에 공개할 예정이다. 
                        지역별 의료 서비스 향상과 소규모 의과대학의 발전을 목표로, 지역 중심의 국립대학 및 소형 의과대학의 
                        입학 정원이 최소한 두 배 가량 확대될 것으로 보인다.
                        """),
        "output": "의대 | 정원 | 확대"
    },
    {
        "input": dedent("""
                        세계보건기구(WHO)는 최근 새로운 건강 위기에 대응하기 위해 국제 협력의 중요성을 강조했다. 
                        전염병 대응 역량의 강화와 글로벌 보건 시스템의 개선이 필요하다고 발표했다.
                        """),
        "output": "세계보건기구 | 건강위기 | 국제"
    },
    {
        "input": dedent("""
                        한국은행 금융통화위원회는 오늘 기준금리를 현재의 연 3.50% 수준에서 동결하기로 결정했다.
                        소비자물가 상승률이 둔화세를 보이고 있으나, 가계부채 증가세와 환율 변동성을 지켜볼 필요가 있다는 판단이다.
                        """),
        "output": "한국은행 | 금리 | 동결"
    }
]

# 각 예시를 포맷팅할 프롬프트 템플릿
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("assistant", "{output}")
])

# Few-shot 프롬프트 템플릿 생성
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,      # 예시 포맷팅 템플릿
    examples=examples                   # 예시 데이터 리스트 -> 예시 포맷팅 템플릿에 적용
)

pprint(few_shot_prompt.invoke({}).to_messages())

[HumanMessage(content='\n정부는 의과대학 입학 정원을 2000명 증가시킬 계획의 세부사항을 이달 20일에 공개할 예정이다. \n지역별 의료 서비스 향상과 소규모 의과대학의 발전을 목표로, 지역 중심의 국립대학 및 소형 의과대학의 \n입학 정원이 최소한 두 배 가량 확대될 것으로 보인다.\n', additional_kwargs={}, response_metadata={}),
 AIMessage(content='의대 | 정원 | 확대', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='\n세계보건기구(WHO)는 최근 새로운 건강 위기에 대응하기 위해 국제 협력의 중요성을 강조했다. \n전염병 대응 역량의 강화와 글로벌 보건 시스템의 개선이 필요하다고 발표했다.\n', additional_kwargs={}, response_metadata={}),
 AIMessage(content='세계보건기구 | 건강위기 | 국제', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='\n한국은행 금융통화위원회는 오늘 기준금리를 현재의 연 3.50% 수준에서 동결하기로 결정했다.\n소비자물가 상승률이 둔화세를 보이고 있으나, 가계부채 증가세와 환율 변동성을 지켜볼 필요가 있다는 판단이다.\n', additional_kwargs={}, response_metadata={}),
 AIMessage(content='한국은행 | 금리 | 동결', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [18]:
# 최종 프롬프트 템플릿 생성
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 뉴스 텍스트에서 핵심 키워드를 추출하는 전문가입니다."),
    few_shot_prompt,
    ("human", "{input}")
])

# 프롬프트 템플릿 출력
pprint(final_prompt.invoke({"input": "뉴스 기사입니다"}).to_messages())

[SystemMessage(content='당신은 뉴스 텍스트에서 핵심 키워드를 추출하는 전문가입니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='\n정부는 의과대학 입학 정원을 2000명 증가시킬 계획의 세부사항을 이달 20일에 공개할 예정이다. \n지역별 의료 서비스 향상과 소규모 의과대학의 발전을 목표로, 지역 중심의 국립대학 및 소형 의과대학의 \n입학 정원이 최소한 두 배 가량 확대될 것으로 보인다.\n', additional_kwargs={}, response_metadata={}),
 AIMessage(content='의대 | 정원 | 확대', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='\n세계보건기구(WHO)는 최근 새로운 건강 위기에 대응하기 위해 국제 협력의 중요성을 강조했다. \n전염병 대응 역량의 강화와 글로벌 보건 시스템의 개선이 필요하다고 발표했다.\n', additional_kwargs={}, response_metadata={}),
 AIMessage(content='세계보건기구 | 건강위기 | 국제', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='\n한국은행 금융통화위원회는 오늘 기준금리를 현재의 연 3.50% 수준에서 동결하기로 결정했다.\n소비자물가 상승률이 둔화세를 보이고 있으나, 가계부채 증가세와 환율 변동성을 지켜볼 필요가 있다는 판단이다.\n', additional_kwargs={}, response_metadata={}),
 AIMessage(content='한국은행 | 금리 | 동결', additional_kwargs={}, response_meta

In [20]:
# 키워드 추출 체인 생성
#chain = final_prompt | llm | StrOutputParser()
chain = final_prompt | ChatOpenAI(model="gpt-4.1-mini") | StrOutputParser() 

# 키워드 추출 체인 실행
result = chain.invoke({
    "input": dedent("""삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다. 
                    이는 AI 반도체 시장에서 지배적인 위치를 차지하고 있는 엔비디아의 독점을 도전하고, 
                    세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 삼성전자의 노력으로 해석된다.""")
})

print(result)

삼성전자 | 인공지능 가속기 | AI 반도체 시장 | 엔비디아 독점 | 반도체 제조업체


In [21]:
import json
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from textwrap import dedent

# 1. 실제 산업 수준의 정교한 예시 정의 (에러 케이스, 성공 케이스 포함)
my_examples = [
    {
        "input": "지난주에 주문한 에어팟 프로(주문번호: 2026-991A) 아직 배송 시작도 안 했던데 언제 오나요?",
        "output": dedent("""\
            {
                "category": "SHIPPING_INQUIRY",
                "priority": "MEDIUM",
                "extracted_entities": {
                    "product_name": "에어팟 프로",
                    "order_id": "2026-991A"
                },
                "requires_api_call": true,
                "api_endpoint": "/api/v1/shipping/status"
            }""")
    },
    {
        "input": "배송받은 구두 사이즈가 생각보다 작아서 270으로 바꾸고 싶어요.",
        "output": dedent("""\
            {
                "category": "EXCHANGE_REQUEST",
                "priority": "HIGH",
                "extracted_entities": {
                    "product_name": "구두",
                    "target_size": "270"
                },
                "requires_api_call": true,
                "api_endpoint": "/api/v1/orders/exchange"
            }""")
    },
    {
        "input": "로그인이 갑자기 안 되는데 사이트 점검 중인가요?",
        "output": dedent("""\
            {
                "category": "TECHNICAL_ISSUE",
                "priority": "HIGH",
                "extracted_entities": {},
                "requires_api_call": false,
                "api_endpoint": null
            }""")
    }
]



In [22]:
# 2. 메시지 구조에 맞는 Example Prompt Template 구성
my_example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("assistant", "{output}")
])
# 3. Few-shot Chat Message Prompt Template 생성
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=my_example_prompt,
    examples=my_examples
)

In [23]:
# 4. 시스템 지시문과 결합한 최종 Prompt Template 구성
# (이때 Assistant가 반드시 순수 JSON만 출력하도록 지시하고 예외 가이드를 정의합니다)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", dedent("""\
        당신은 고객 센터의 수신 문의를 자동 분류하고 데이터를 추출하는 미들웨어 엔진입니다.
        사용자의 입력을 분석하여 카테고리, 우선순위, 엔티티를 파싱하고 JSON 형태로만 답변해야 합니다.
        답변에는 마크다운 포맷(```json ...)을 포함하지 말고 오직 순수 JSON 텍스트만 출력해야 합니다.""")),
    few_shot_prompt,
    ("human", "{input}")
])

# 5. LLM 및 체인 구성 (실제 환경에서는 구조화된 출력을 위해 temperature를 낮춥니다)
llm = ChatOpenAI(
    model='gpt-4.1-nano',
    temperature=0.0  # 일관성을 위해 0으로 설정
)

chain = final_prompt | llm | StrOutputParser()

# 6. 실전 문의사항 테스트
test_input = "안녕하세요, 어제 산 스웨터(주문코드: 88712) 환불 신청하고 싶습니다."
result_raw = chain.invoke({"input": test_input})

# 7. 추출 결과 로깅 및 파싱 테스트
print("--- [LLM Raw Response] ---")
print(result_raw)

# 실제 서비스에서는 JSON 변환 후 후속 비즈니스 로직(API 호출 등)을 수행합니다.
parsed_result = json.loads(result_raw)
print("\n--- [Parsed Backend Dict] ---")
print(f"분류 카테고리: {parsed_result['category']}")
print(f"우선 순위: {parsed_result['priority']}")
print(f"추출 정보: {parsed_result['extracted_entities']}")
print(f"API 호출 엔드포인트: {parsed_result['api_endpoint']}")

--- [LLM Raw Response] ---
{
    "category": "REFUND_REQUEST",
    "priority": "HIGH",
    "extracted_entities": {
        "product_name": "스웨터",
        "order_code": "88712"
    },
    "requires_api_call": true,
    "api_endpoint": "/api/v1/orders/refund"
}

--- [Parsed Backend Dict] ---
분류 카테고리: REFUND_REQUEST
우선 순위: HIGH
추출 정보: {'product_name': '스웨터', 'order_code': '88712'}
API 호출 엔드포인트: /api/v1/orders/refund


`(4) Dynamic Few-Shot Prompting`

* **Dynamic Few-Shot Prompting**은 상황에 따라 적절한 예시를 동적으로 선택하여 사용하는 고급 프롬프팅 기법으로, **BaseExampleSelector**를 통해 입력값과 가장 연관성이 높은 예시들을 자동으로 선별합니다.

* 대표적으로 **SemanticSimilarityExampleSelector**는 의미적 유사도를 기반으로 예시를 선택하며, 이를 통해 주어진 입력 상황에 가장 적합한 예시들만을 효율적으로 활용할 수 있습니다.

* **example_prompt**를 통해 선택된 예시들을 AI 시스템이 이해하기 쉬운 형태(예: human-AI 대화 , human-function call)로 변환하여 더 효과적인 학습과 응답 생성이 가능하게 합니다.


- **장점**

    - 상황에 맞는 가장 연관성 높은 예시만을 선택적으로 활용할 수 있다
    - 프롬프트의 길이를 효율적으로 관리할 수 있다
    - 응답의 일관성과 품질을 향상시킬 수 있다

In [24]:
from langchain_ollama import OllamaEmbeddings 
from langchain_core.vectorstores import InMemoryVectorStore

# 고객 문의 유형별 응대 예시를 준비
examples = [
    {
        "input": "환불 절차가 어떻게 되나요?",
        "output": "환불 절차는 다음과 같습니다:\n1. 구매내역에서 환불을 신청해주세요\n2. 반품 상품을 발송해주세요\n3. 상품 검수 후 3-5일 내 환불이 완료됩니다"
    },
    {
        "input": "배송이 늦어지고 있어요", 
        "output": "불편을 드려 죄송합니다. 주문번호를 알려주시면 배송 상태를 즉시 확인해드리겠습니다."
    },
    {
        "input": "옷 사이즈가 안 맞아요",
        "output": "사이즈 교환은 무료로 진행됩니다. 교환 신청 후 동일 상품의 다른 사이즈로 발송해드리겠습니다."
    },
    {
        "input": "제품이 불량이에요",
        "output": "불편을 드려 대단히 죄송합니다. 불량 부분 사진과 함께 1:1 문의에 접수해주시면 빠르게 처리해드리겠습니다."
    }
]

# 예시 데이터를 벡터화할 텍스트로 변환
to_vectorize = [" ".join(example.values()) for example in examples]

# Ollama 임베딩 모델 생성
embeddings = OllamaEmbeddings(model="bge-m3")

# 벡터 스토어 생성
vector_store = InMemoryVectorStore.from_texts(
    to_vectorize,    # 벡터화할 텍스트 리스트
    embeddings,      # 임베딩 모델
    metadatas=examples    # 메타데이터: 예시 데이터
    )

# VectorStore에 저장된 Document 개수 확인
print(f"VectorStore에 저장된 Document 개수: {len(vector_store.store.keys())}")

VectorStore에 저장된 Document 개수: 4


In [26]:
vector_store.similarity_search("불량", k=1)

[Document(id='f7a0cbe1-40e1-4b50-b8b9-1d41a091a7f1', metadata={'input': '제품이 불량이에요', 'output': '불편을 드려 대단히 죄송합니다. 불량 부분 사진과 함께 1:1 문의에 접수해주시면 빠르게 처리해드리겠습니다.'}, page_content='제품이 불량이에요 불편을 드려 대단히 죄송합니다. 불량 부분 사진과 함께 1:1 문의에 접수해주시면 빠르게 처리해드리겠습니다.')]

In [25]:
from langchain_core.example_selectors import SemanticSimilarityExampleSelector

# 유사한 2개의 예시를 선택하는 selector 생성
example_selector = SemanticSimilarityExampleSelector(
    vectorstore=vector_store,
    k=2
)

# 선택된 예시 확인
selected_examples = example_selector.select_examples({"input":"상품이 파손되어 왔어요"})
pprint(selected_examples)

[{'input': '제품이 불량이에요',
  'output': '불편을 드려 대단히 죄송합니다. 불량 부분 사진과 함께 1:1 문의에 접수해주시면 빠르게 처리해드리겠습니다.'},
 {'input': '환불 절차가 어떻게 되나요?',
  'output': '환불 절차는 다음과 같습니다:\n'
            '1. 구매내역에서 환불을 신청해주세요\n'
            '2. 반품 상품을 발송해주세요\n'
            '3. 상품 검수 후 3-5일 내 환불이 완료됩니다'}]


In [27]:
# 챗봇 프롬프트 템플릿 생성
few_shot_prompt = FewShotChatMessagePromptTemplate(
    input_variables=["input"],
    example_selector=example_selector,
    example_prompt=ChatPromptTemplate.from_messages([
        ("human", "{input}"),
        ("assistant", "{output}")
    ])
)

pprint(few_shot_prompt.invoke({"input": "상품이 파손되어 왔어요"}).to_messages())

[HumanMessage(content='제품이 불량이에요', additional_kwargs={}, response_metadata={}),
 AIMessage(content='불편을 드려 대단히 죄송합니다. 불량 부분 사진과 함께 1:1 문의에 접수해주시면 빠르게 처리해드리겠습니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='환불 절차가 어떻게 되나요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='환불 절차는 다음과 같습니다:\n1. 구매내역에서 환불을 신청해주세요\n2. 반품 상품을 발송해주세요\n3. 상품 검수 후 3-5일 내 환불이 완료됩니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [28]:
# 최종 프롬프트 생성 
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절하고 전문적인 고객 서비스 담당자입니다."),
    few_shot_prompt,
    ("human", "{input}")
])

pprint(final_prompt.invoke({"input": "상품이 파손되어 왔어요"}).to_messages())

[SystemMessage(content='당신은 친절하고 전문적인 고객 서비스 담당자입니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='제품이 불량이에요', additional_kwargs={}, response_metadata={}),
 AIMessage(content='불편을 드려 대단히 죄송합니다. 불량 부분 사진과 함께 1:1 문의에 접수해주시면 빠르게 처리해드리겠습니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='환불 절차가 어떻게 되나요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='환불 절차는 다음과 같습니다:\n1. 구매내역에서 환불을 신청해주세요\n2. 반품 상품을 발송해주세요\n3. 상품 검수 후 3-5일 내 환불이 완료됩니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='상품이 파손되어 왔어요', additional_kwargs={}, response_metadata={})]


In [29]:
# 챗봇 체인 생성
chain = final_prompt | llm | StrOutputParser()

# 체인 실행
response = chain.invoke({
    "input": "상품이 파손되어 왔어요"
})

pprint(response)

('불편을 드려 정말 죄송합니다. 파손된 상품의 사진을 함께 보내주시면 빠른 처리에 도움이 됩니다. 1:1 문의를 통해 접수해주시면 신속하게 '
 '해결해드리겠습니다.')


---

## 실습 프로젝트

### 감정 분석기 만들기

- Zero-shot 방식으로 텍스트의 감정(긍정/부정)을 분류하는 프롬프트를 작성합니다.
- Few-shot 방식으로 3개의 예시를 포함하여 같은 작업을 수행하는 프롬프트를 작성합니다.
- LLM 체인을 구성하여 두 방식의 출력 결과를 비교합니다.

In [30]:
# Zero-shot 방식 프롬프트
# 1. PromptTemplate을 사용하여 감정 분석 프롬프트 작성
# 2. 입력 변수: text
# 3. 지시사항: 텍스트의 감정을 '긍정' 또는 '부정'으로 분류
# 4. 체인 구성: prompt | llm | StrOutputParser()

# 테스트 데이터
test_texts = [
    "이 제품 정말 마음에 들어요! 강력 추천합니다.",
    "배송이 너무 늦고 제품도 불량이네요. 최악입니다."
]

# Zero-shot 프롬프트 템플릿 생성
zero_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template="다음의 텍스트의 감정을 긍정 또는 부정으로 분류해주세요. {text}"
)

# 체인 생성
chain = zero_shot_prompt | llm | StrOutputParser()

# Zero-shot 실행
for text in enumerate(test_texts):
    zero_shot_result = chain.invoke(input={"text": text}) 
    print(zero_shot_result)


긍정 (1)
부정


In [32]:
# Few-shot 방식 프롬프트
# 1. FewShotChatMessagePromptTemplate 또는 PromptTemplate 사용
# 2. 3개의 예시 포함 (긍정 2개, 부정 1개 권장)
# 3. 각 예시는 {"input": "텍스트", "output": "긍정/부정"} 형태
# 4. 체인 구성 및 동일한 테스트 데이터로 실행
# 5. Zero-shot 결과와 비교

# 예시 데이터 힌트
examples = [
    {"input": "배송이 빠르고 품질이 좋아요!", "output": "긍정"},
    {"input": "배송이 늦게오고, 상품이 망가져있어요", "output": "부정"},
    {"input": "상품 상태가 좋아요 잘활용하고 있어요~~", "output": "긍정"},
]

# 각 예시를 포맷팅할 프롬프트 템플릿
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("assistant", "{output}")
])

# Few-shot 프롬프트 템플릿 생성
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,      # 예시 포맷팅 템플릿
    examples=examples                   # 예시 데이터 리스트 -> 예시 포맷팅 템플릿에 적용
)

pprint(few_shot_prompt.invoke({}).to_messages())

# 여기에 코드를 작성하세요
# 최종 프롬프트 템플릿 생성
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음의 텍스트의 감정을 긍정 또는 부정으로 분류해주세요. "),
    few_shot_prompt,
    ("human", "{input}")
])

chain = final_prompt | llm | StrOutputParser()

result = chain.invoke({"input": "젊댉삵짉막셀욝"});

pprint(result)

[HumanMessage(content='배송이 빠르고 품질이 좋아요!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='긍정', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='배송이 늦게오고, 상품이 망가져있어요', additional_kwargs={}, response_metadata={}),
 AIMessage(content='부정', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='상품 상태가 좋아요 잘활용하고 있어요~~', additional_kwargs={}, response_metadata={}),
 AIMessage(content='긍정', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
'부정'


In [35]:
result = chain.invoke({"input": "고민하고 있던건데 바로 구입했어요~~젊댉삵짉막셀욝 ㅈㄴ덝럼웤욬"});
pprint(result)

'긍정'


In [34]:
pprint(result)

'부정'
